# R1 on Kaggle: redesigned pilot of the repair test (train and dev templates)

Pre-registration: `main-study/prereg/R1.md` (DECISIONS D48). One **Save & Run All (Commit)**, 2 × T4,
one worker per GPU. Inputs: the private dataset built by `python -m repair.build_inputs`
(items.jsonl, vectors.npy, meta.json; train and dev templates only) and Kaggle Models
`google/gemma-2/transformers/gemma-2-2b-it/2`.

Order: stability samples → calibration (oracle on dev failures over the grid) → m* →
self / evaluation / 10 random at m*/2 and m* → projection (budget gate) → the pilot at m* → report.
**Cap: 8 T4-hours = 4 session-hours on 2 × T4**, setup included; a hard stop ends any stage cleanly.
Attach this kernel's earlier output as input to resume.

In [ ]:
# 0. Settings
REPO_URL = 'https://github.com/mehul24d/Reasoning-Failure-Diagnosis.git'
BRANCH = 'main'
CAP_T4H = 8.0            # all GPU work of the 2026-11-26 request (prereg/R1.md)
USED_T4H_BEFORE = 0.0    # T4-hours already used by earlier sessions of this request
RESERVE_MIN = 8          # kept free before the hard stop for status files


In [ ]:
# 1. Session state (re-run at the top of every cell)
import json
SESSION = r'''
import os, sys, time, json, glob, shutil, subprocess, hashlib
WORK = '/kaggle/working/btp'
OUT = WORK + '/r1'
INP = WORK + '/r1_inputs'
REPO = '/tmp/Reasoning-Failure-Diagnosis'
MS = REPO + '/main-study'
VENV = '/tmp/venv'
os.makedirs(OUT, exist_ok=True)
os.environ.update(HF_HUB_OFFLINE='1', TOKENIZERS_PARALLELISM='false',
                  PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
                  PATH=VENV + '/bin:' + os.environ['PATH'], VIRTUAL_ENV=VENV, PYTHONPATH=MS)
if not os.path.exists('/tmp/btp_session_start'):
    open('/tmp/btp_session_start', 'w').write(str(time.time()))
T0 = float(open('/tmp/btp_session_start').read())
S = json.load(open('/tmp/r1_settings.json'))
HARD_STOP = T0 + (S['CAP_T4H'] - S['USED_T4H_BEFORE']) / 2 * 3600 - S['RESERVE_MIN'] * 60
def used_t4h():
    return S['USED_T4H_BEFORE'] + 2 * (time.time() - T0) / 3600
def sh(cmd, ok=(0,)):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    if rc not in ok:
        raise RuntimeError(f'command failed ({rc}): {cmd}')
    return rc
def workers(args, n=2):
    # one process per T4; returns the exit codes
    ps = [subprocess.Popen(f'cd {MS} && {VENV}/bin/python -m repair.run --inputs {INP} --out {OUT} '
                           f'{args} --worker {w} --n-workers {n} --deadline {HARD_STOP}',
                           shell=True, stdout=open(f'{OUT}/log_w{w}.txt', 'a'), stderr=subprocess.STDOUT)
          for w in range(n)]
    codes = [p.wait() for p in ps]
    for w in range(n):
        print(open(f'{OUT}/log_w{w}.txt').read()[-1500:])
    if any(codes):
        raise RuntimeError(f'worker exit codes {codes}')
    return codes
def log(stage, **kw):
    with open(f'{OUT}/session_log.jsonl', 'a') as f:
        f.write(json.dumps({'stage': stage, 'time': time.time(), 'used_t4h': round(used_t4h(), 3), **kw}) + '\n')
'''
open('/tmp/r1_settings.json', 'w').write(json.dumps(dict(CAP_T4H=CAP_T4H, USED_T4H_BEFORE=USED_T4H_BEFORE,
                                                         RESERVE_MIN=RESERVE_MIN)))
open('/tmp/r1_session.py', 'w').write(SESSION)
exec(SESSION)
print('hard stop at', time.strftime('%H:%M:%S', time.localtime(HARD_STOP)))
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')


In [ ]:
exec(open('/tmp/r1_session.py').read())
# 2. Code, environment (main-study/requirements.txt, Python 3.13) and the Kaggle Models weights
if not os.path.isdir(REPO):
    sh(f'git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}')
sh(f'git -C {REPO} log --oneline -1')
if not os.path.exists(VENV + '/bin/python'):
    sh(f'{sys.executable} -m pip -q install uv')
    sh(f'uv venv --seed --python 3.13 {VENV}')
    sh(f'uv pip install --python {VENV}/bin/python -r {MS}/requirements.txt')
sh(f'{VENV}/bin/python -m pip freeze > {OUT}/pip_freeze.txt')
cfg = sorted(glob.glob('/kaggle/input/**/gemma-2-2b-it/**/config.json', recursive=True))
if not cfg:
    raise RuntimeError('add the input google/gemma-2 / transformers / gemma-2-2b-it / 2')
src = os.path.dirname(cfg[-1])
rev = hashlib.sha1(src.encode()).hexdigest()
hub = os.path.expanduser('~/.cache/huggingface/hub/models--google--gemma-2-2b-it')
os.makedirs(f'{hub}/snapshots/{rev}', exist_ok=True)
os.makedirs(f'{hub}/refs', exist_ok=True)
for f in os.listdir(src):
    if not os.path.exists(f'{hub}/snapshots/{rev}/{f}'):
        os.symlink(os.path.join(src, f), f'{hub}/snapshots/{rev}/{f}')
open(f'{hub}/refs/main', 'w').write(rev)
print('weights:', src)
open(f'{OUT}/code_commit.txt', 'w').write(subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'],
                                                         capture_output=True, text=True).stdout)


In [ ]:
exec(open('/tmp/r1_session.py').read())
# 3. Inputs (hash-checked against meta.json) and, if attached, an earlier output to resume from
meta_f = sorted(glob.glob('/kaggle/input/**/items.jsonl', recursive=True))
assert meta_f, 'attach the R1 inputs dataset'
d = os.path.dirname(meta_f[0])
os.makedirs(INP, exist_ok=True)
for f in ('items.jsonl', 'vectors.npy', 'meta.json'):
    shutil.copy2(f'{d}/{f}', f'{INP}/{f}')
meta = json.load(open(f'{INP}/meta.json'))
for f, h in meta['sha256'].items():
    assert hashlib.sha256(open(f'{INP}/{f}', 'rb').read()).hexdigest() == h, f
print('inputs ok:', meta['n_failures'], 'failures,', meta['n_solved'], 'solved')
prev = [p for p in glob.glob('/kaggle/input/**/btp/r1', recursive=True) if os.path.isdir(p)]
for p in prev:
    for f in os.listdir(p):
        if not os.path.exists(f'{OUT}/{f}'):
            shutil.copy2(f'{p}/{f}', f'{OUT}/{f}')
print('resumed from', prev or 'nothing')
sh(f'cd {MS} && {VENV}/bin/python -m pytest -q -x tests/test_repair_engine.py tests/test_repair_analyze.py')
log('setup')


In [ ]:
exec(open('/tmp/r1_session.py').read())
# 4. Stability: k = 5 samples at T = 0.7 per failure (S3 settings)
workers('--stage stability')
log('stability')


In [ ]:
exec(open('/tmp/r1_session.py').read())
# 5. Calibration step 1-2: the oracle edit on the dev failures over the whole grid, then m*
GRID = '0.25,0.5,1,2,4,8,16'
workers(f'--stage edit --tag cal_oracle --conds oracle --roles failure --splits dev --mults {GRID}')
sh(f'cd {MS} && {VENV}/bin/python -m repair.analyze calibrate --inputs {INP} --out {OUT}')
cal = json.load(open(f'{OUT}/calibration.json'))
print('m* =', cal['m_star'], '|', cal['reason'])
log('cal_oracle', m_star=cal['m_star'], reason=cal['reason'])


In [ ]:
exec(open('/tmp/r1_session.py').read())
# 6. Calibration step 3: what self-sourced, evaluation and random edits do at m*/2 and m*
#    (no m*: 10 random directions on the dev failures over the whole grid, then STOP)
cal = json.load(open(f'{OUT}/calibration.json'))
m = cal['m_star']
if m is not None:
    ms = ','.join(str(x) for x in [m / 2, m] if x in [0.25, 0.5, 1, 2, 4, 8, 16])
    workers(f'--stage edit --tag cal_rest --conds self,eval,rand0-9 --roles failure,solved --splits dev --mults {ms}')
else:
    workers('--stage edit --tag cal_rest --conds rand0-9 --roles failure --splits dev --mults 0.25,0.5,1,2,4,8,16')
sh(f'cd {MS} && {VENV}/bin/python -m repair.analyze calibrate --inputs {INP} --out {OUT}')
log('cal_rest')


In [ ]:
exec(open('/tmp/r1_session.py').read())
# 7. Budget gate, then the pilot at m* on all train + dev items (dev rows at m* are reused)
cal = json.load(open(f'{OUT}/calibration.json'))
m = cal['m_star']
STATUS = {}
if m is None:
    STATUS['main'] = 'not run: no m* (stop rule of prereg/R1.md)'
else:
    sh(f'cd {MS} && {VENV}/bin/python -m repair.analyze project --inputs {INP} --out {OUT} --used-t4h {used_t4h():.3f}')
    pr = json.load(open(f'{OUT}/projection.json'))
    if not pr['within_budget']:
        STATUS['main'] = f"not run: projected {pr['projected_t4h_with_safety']:.2f} + used {pr['used_t4h']:.2f} > {pr['budget_t4h']}"
    else:
        reuse = f'{OUT}/main_w9.jsonl'
        if not os.path.exists(reuse):
            with open(reuse, 'w') as fo:
                for tag in ('cal_oracle', 'cal_rest'):
                    for p in glob.glob(f'{OUT}/{tag}_w[0-9].jsonl'):
                        for line in open(p):
                            r = json.loads(line)
                            if r['m'] == m:
                                fo.write(line if line.endswith('\n') else line + '\n')
        workers(f'--stage edit --tag main --conds oracle,self,eval,rand0-99 --roles failure,solved --splits train,dev --mults {m}')
        STATUS['main'] = 'complete' if time.time() < HARD_STOP else 'stopped at the hard stop (partial)'
        sh(f'cd {MS} && {VENV}/bin/python -m repair.analyze report --inputs {INP} --out {OUT}')
print(STATUS)
log('main', **STATUS)


In [ ]:
exec(open('/tmp/r1_session.py').read())
# 8. End of the session: T4-hours used, outputs zipped (the session and the quota end here)
s = {'finished': time.strftime('%Y-%m-%d %H:%M:%S'), 'session_hours': round((time.time() - T0) / 3600, 3),
     't4_hours_this_session': round(2 * (time.time() - T0) / 3600, 3), 'used_t4h_total': round(used_t4h(), 3),
     'commit': open(f'{OUT}/code_commit.txt').read().strip()}
json.dump(s, open(f'{OUT}/session_status.json', 'w'), indent=2)
shutil.make_archive('/kaggle/working/r1_results', 'zip', OUT)
print(json.dumps(s, indent=2))
